In [23]:
#imports en connecties
import sqlite3
import pandas as pd
import csv
from datetime import datetime

# connectie met SDM
sdm_conn = sqlite3.connect("BikeToDriveDatabase.db")

# connecties met bron databases 
accessoireverkoop_conn = sqlite3.connect("BikeToDrive_1_Accessoireverkoop.db")
fietsverkoop_conn = sqlite3.connect("BikeToDrive_2_Fietsverkoop.db")
onderhoud_conn = sqlite3.connect("BikeToDrive_3_Onderhoud.db")
accessoireinkoop_conn = sqlite3.connect("BikeToDrive_4_Accessoire_Inkoop.db")
fietsinkoop_conn = sqlite3.connect("BikeToDrive_5_Fiets_Inkoop.db")

# logfile voor sdm
sdm_log_file = "sdm_log.csv"

# 1 keer uitvoeren: header maken
with open(sdm_log_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow([
        "timestamp", "stap", "tabel", "actie", "aantal", "status", "melding"
    ])

# functie om sdm logging weg te schrijven
def log_sdm(stap, tabel, actie, aantal, status, melding=""):
    with open(sdm_log_file, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            stap,
            tabel,
            actie,
            aantal,
            status,
            melding
        ])

In [22]:
# als er iets mis is met de connection of foutjes in database run dit zodat de connection wordt gestopt en je opnieuw kan proberen !

sdm_conn.close()

In [24]:
# inlaadstrategie (reset) full refresh
# eerste alle tabellen leegmaken

cursor = sdm_conn.cursor()

tables = [
    "Onderhoud", "Onderhoud_Monteur", "Onderhoud_Fiets", "Onderhoud_Filiaal", "Onderhoud_Fabrikant",
    "Accessoire_Inkoop", "Accessoire_Inkoop_Accessoire", "Accessoire_Inkoop_Leverancier",
    "Fiets_Inkoop", "Fiets_Inkoop_Fiets", "Fiets_Inkoop_Fabrikant",
    "Accessoire_Verkoop", "Accessoire_Verkoop_Klant", "Accessoire_Verkoop_Accessoire",
    "Accessoire_Verkoop_Monteur", "Accessoire_Verkoop_Filiaal", "Accessoire_Verkoop_Leverancier",
    "Fiets_Verkoop", "Fiets_Verkoop_Klant", "Fiets_Verkoop_Fiets",
    "Fiets_Verkoop_Monteur", "Fiets_Verkoop_Filiaal", "Fiets_Verkoop_Fabrikant"
]

for table in tables:
    cursor.execute(f"DELETE FROM {table}")

sdm_conn.commit()

log_sdm("FULL_REFRESH", "SDM", "delete_all", len(tables), "OK", "alle sdm tabellen geleegd")


In [17]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""", sdm_conn)

,name
0,Accessoire_Inkoop
1,Accessoire_Inkoop_Accessoire
2,Accessoire_Inkoop_Leverancier
3,Accessoire_Verkoop
4,Accessoire_Verkoop_Accessoire
5,Accessoire_Verkoop_Filiaal
6,Accessoire_Verkoop_Klant
7,Accessoire_Verkoop_Leverancier
8,Accessoire_Verkoop_Monteur
9,Fiets_Inkoop


In [25]:
# dan data inladen (extract + load)

# iedere database die apart was heeft een apart blokje om te runnen zodat het netter blijft.
# data inladen voor alles met Onderhoud
# LINQ (dus SQL query in python)
# tabellen andersom inladen door foreign key relaties

# eerst data inladen van bron tabel daarna data inladen van sdm tabel
# read_sql_query("bron_tabel", bron_conn)
# to_sql("SDM_tabel", sdm_conn)

# onderhoud data inladen

# Fabrikant
df = pd.read_sql_query("SELECT * FROM Fabrikant", onderhoud_conn)
df.to_sql("Onderhoud_Fabrikant", sdm_conn, if_exists="append", index=False)

log_sdm("LOAD", "Onderhoud_Fabrikant", "insert", len(df), "OK")

# Filiaal
df = pd.read_sql_query("SELECT * FROM Filiaal", onderhoud_conn)
df.to_sql("Onderhoud_Filiaal", sdm_conn, if_exists="append", index=False)

log_sdm("LOAD", "Onderhoud_Filiaal", "insert", len(df), "OK")

# Fiets
df = pd.read_sql_query("SELECT * FROM Fiets", onderhoud_conn)
df.to_sql("Onderhoud_Fiets", sdm_conn, if_exists="append", index=False)

log_sdm("LOAD", "Onderhoud_Fiets", "insert", len(df), "OK")

# Monteur
df = pd.read_sql_query("SELECT * FROM Monteur", onderhoud_conn)
df.to_sql("Onderhoud_Monteur", sdm_conn, if_exists="append", index=False)

log_sdm("LOAD", "Onderhoud_Monteur", "insert", len(df), "OK")

# Onderhoud
df = pd.read_sql_query("SELECT * FROM Onderhoud", onderhoud_conn)
df.to_sql("Onderhoud", sdm_conn, if_exists="append", index=False)

log_sdm("LOAD", "Onderhoud", "insert", len(df), "OK")

In [26]:
# accessoire inkoop data inladen

# Leverancier
df = pd.read_sql_query("SELECT * FROM Leverancier", accessoireinkoop_conn)
df.to_sql("Accessoire_Inkoop_Leverancier", sdm_conn, if_exists="append", index=False)
log_sdm("LOAD", "Accessoire_Inkoop_Leverancier", "insert", len(df), "OK")

# Accessoire
df = pd.read_sql_query("SELECT * FROM Accessoire", accessoireinkoop_conn)
df.to_sql("Accessoire_Inkoop_Accessoire", sdm_conn, if_exists="append", index=False)
log_sdm("LOAD", "Accessoire_Inkoop_Accessoire", "insert", len(df), "OK")

# Inkoop
df = pd.read_sql_query("SELECT * FROM Accessoire_Inkoop", accessoireinkoop_conn)
df.to_sql("Accessoire_Inkoop", sdm_conn, if_exists="append", index=False)
log_sdm("LOAD", "Accessoire_Inkoop", "insert", len(df), "OK")

In [27]:
# data inladen accessoireverkoop

# Klant
df = pd.read_sql_query("SELECT * FROM Klant", accessoireverkoop_conn)
df.to_sql("Accessoire_Verkoop_Klant", sdm_conn, if_exists="append", index=False)
log_sdm("LOAD", "Accessoire_Verkoop_Klant", "insert", len(df), "OK")

# Accessoire
df = pd.read_sql_query("SELECT * FROM Accessoire", accessoireverkoop_conn)
df.to_sql("Accessoire_Verkoop_Accessoire", sdm_conn, if_exists="append", index=False)
log_sdm("LOAD", "Accessoire_Verkoop_Accessoire", "insert", len(df), "OK")

# Monteur
df = pd.read_sql_query("SELECT * FROM Monteur", accessoireverkoop_conn)
df.to_sql("Accessoire_Verkoop_Monteur", sdm_conn, if_exists="append", index=False)
log_sdm("LOAD", "Accessoire_Verkoop_Monteur", "insert", len(df), "OK")

# filiaal
df = pd.read_sql_query("SELECT * FROM Filiaal", accessoireverkoop_conn)
df.to_sql("Accessoire_Verkoop_Filiaal", sdm_conn, if_exists="append", index=False)
log_sdm("LOAD", "Accessoire_Verkoop_Filiaal", "insert", len(df), "OK")

# leverancier
df = pd.read_sql_query("SELECT * FROM Leverancier", accessoireverkoop_conn)
df.to_sql("Accessoire_Verkoop_Leverancier", sdm_conn, if_exists="append", index=False)
log_sdm("LOAD", "Accessoire_Verkoop_Leverancier", "insert", len(df), "OK")

# Verkoop
df = pd.read_sql_query("SELECT * FROM Accessoire_Verkoop", accessoireverkoop_conn)
df.to_sql("Accessoire_Verkoop", sdm_conn, if_exists="append", index=False)
log_sdm("LOAD", "Accessoire_Verkoop", "insert", len(df), "OK")

In [21]:
cursor = sdm_conn.cursor()

drop_tables = [
    "Onderhoud",
    "Onderhoud_Monteur",
    "Onderhoud_Fiets",
    "Onderhoud_Filiaal",
    "Onderhoud_Fabrikant",
    "Accessoire_Inkoop",
    "Accessoire_Inkoop_Accessoire",
    "Accessoire_Inkoop_Leverancier",
    "Fiets_Inkoop",
    "Fiets_Inkoop_Fiets",
    "Fiets_Inkoop_Fabrikant",
    "Accessoire_Verkoop",
    "Accessoire_Verkoop_Klant",
    "Accessoire_Verkoop_Accessoire",
    "Accessoire_Verkoop_Monteur",
    "Accessoire_Verkoop_Filiaal",
    "Accessoire_Verkoop_Leverancier",
    "Fiets_Verkoop",
    "Fiets_Verkoop_Klant",
    "Fiets_Verkoop_Fiets",
    "Fiets_Verkoop_Monteur",
    "Fiets_Verkoop_Filiaal",
    "Fiets_Verkoop_Fabrikant"
]

for table in drop_tables:
    cursor.execute(f"DROP TABLE IF EXISTS {table}")

sdm_conn.commit()

print("alle sdm tabellen zijn verwijderd")

alle sdm tabellen zijn verwijderd


In [28]:
# data inladen fietsverkoop

# klant
df = pd.read_sql_query("SELECT * FROM Klant", fietsverkoop_conn)
df.to_sql("Fiets_Verkoop_Klant", sdm_conn, if_exists="append", index=False)
log_sdm("LOAD", "Fiets_Verkoop_Klant", "insert", len(df), "OK")

# fiets
df = pd.read_sql_query("SELECT * FROM Fiets", fietsverkoop_conn)
df.to_sql("Fiets_Verkoop_Fiets", sdm_conn, if_exists="append", index=False)
log_sdm("LOAD", "Fiets_Verkoop_Fiets", "insert", len(df), "OK")

# monteur
df = pd.read_sql_query("SELECT * FROM Monteur", fietsverkoop_conn)
df.to_sql("Fiets_Verkoop_Monteur", sdm_conn, if_exists="append", index=False)
log_sdm("LOAD", "Fiets_Verkoop_Monteur", "insert", len(df), "OK")

# fabrikant
df = pd.read_sql_query("SELECT * FROM Fabrikant", fietsverkoop_conn)
df.to_sql("Fiets_Verkoop_Fabrikant", sdm_conn, if_exists="append", index=False)
log_sdm("LOAD", "Fiets_Verkoop_Fabrikant", "insert", len(df), "OK")

# filiaal
df = pd.read_sql_query("SELECT * FROM Filiaal", fietsverkoop_conn)
df.to_sql("Fiets_Verkoop_Filiaal", sdm_conn, if_exists="append", index=False)
log_sdm("LOAD", "Fiets_Verkoop_Filiaal", "insert", len(df), "OK")

# fietsverkoop
df = pd.read_sql_query("SELECT * FROM Fiets_Verkoop", fietsverkoop_conn)
df.to_sql("Fiets_Verkoop", sdm_conn, if_exists="append", index=False)
log_sdm("LOAD", "Fiets_Verkoop", "insert", len(df), "OK")

In [29]:
pd.read_sql_query("SELECT * FROM Fiets_Verkoop_Filiaal;", sdm_conn) #controleren of data juist is geladen per tabel

log_sdm("CHECK", "Fiets_Verkoop_Filiaal", "select_check", 0, "OK", "controlequery uitgevoerd")